In [8]:
import pandas as pd
from reusable import label_Encoder,oneHot_encoder, cleaning, binning, flag, bmi_category
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
health = pd.read_csv('insurance.csv')

df = pd.DataFrame(health)



df = label_Encoder(df)
df = oneHot_encoder(df)
df = cleaning(df)
df = binning(df)
df = flag(df)
df = bmi_category(df)

df.columns


Index(['age', 'sex', 'bmi', 'children', 'smoker', 'charges',
       'region_northwest', 'region_southeast', 'region_southwest',
       'high_risks', 'obese', 'overweight', 'underweight'],
      dtype='object')

In [16]:
X = df.drop(columns=['charges','children','underweight','overweight'])
Y = df['charges']

X_train,X_test,Y_train,Y_test = train_test_split(X,Y, train_size=0.8, test_size=0.2,random_state=42)
model = Ridge()

params = {
    'alpha':[0.001,0.01,0.1,1,10,100,1000]
}

gs_cv = GridSearchCV(model, params, cv=5,scoring='r2')
gs_cv.fit(X_train,Y_train)
print("best alpha(grid search):",gs_cv.best_params_)
print("best r2(cv)",gs_cv.best_score_)
print('best model estimator',gs_cv.best_estimator_)
model.fit(X_train,Y_train)
y_pred = model.predict(X_test)

mae_r = mean_absolute_error(Y_test,y_pred)
mse_r = mean_squared_error(Y_test,y_pred)
r2_r = r2_score(Y_test,y_pred)

print("\nmean absolute error:",mae_r)
print('mean square error,:',mse_r)
print("r2_score:",r2_r)





best alpha(grid search): {'alpha': 1}
best r2(cv) 0.8463027038708375
best model estimator Ridge(alpha=1)

mean absolute error: 2419.6982811872026
mean square error,: 17353034.852781422
r2_score: 0.9055648315970667


In [ ]:
model_l = Lasso()

model_l.fit(X_train,Y_train)

gs_cv = GridSearchCV(model_l,params,cv=10,scoring='r2')
gs_cv.fit(X_train,Y_train)

best_model = gs_cv.best_estimator_


y_pred = best_model.predict(X_test)

# print(y_pred)
mae_l = mean_absolute_error(Y_test,y_pred)
mse_l = mean_squared_error(Y_test,y_pred)
r2_l = r2_score(Y_test,y_pred)

print("\nmean absolute error:",mae_l)
print('mean square error,:',mse_l)
print("r2_score:",r2_l)
#r2_score: 0.9052346102383113 -> without gs_cv with only model_l.predict(X_test)
#r2_score: 0.9058106200447934 -> with gs_cv when cv = 10(when cv was 5 the r2 were a bit less)


mean absolute error: 2420.3685552791576
mean square error,: 17413715.029928617
r2_score: 0.9052346102383113
